In [ ]:
counts = labelled["category"].value_counts()
print(counts)

keep = counts[counts >= 5].index
labelled = labelled[labelled["category"].isin(keep)]


In [ ]:
import sqlite3

from sklearn.model_selection import train_test_split

from spending_analyser.storage import load_transactions

conn = sqlite3.connect("../data/spending.db")
df = load_transactions(conn)
labelled = df[df["category_source"] == "manual"]
counts = labelled["category"].value_counts()
keep = counts[counts >= 5].index
labelled = labelled[labelled["category"].isin(keep)]

train, test = train_test_split(
    labelled,
    test_size=0.25,
    random_state=42,
    stratify=labelled["category"],
)

print(len(train), "trainig rows,", len(test), "test rows")

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(train["description"])
X_test = vectorizer.transform(test["description"])

y_train = train["category"]
y_test = test["category"]

print(X_train.shape)
print(vectorizer.get_feature_names_out()[:20])

In [ ]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train, y_train)

predictions = model.predict(X_test)
accuracy = (predictions == y_test).mean()

print(f"Model accuracy on test set: {accuracy:.0%}")

In [ ]:
from spending_analyser.categoriser import categorise

rule_preds = test.apply(lambda row: categorise(row["description"], row["amount"]), axis=1)
rule_accuracy = (rule_preds == y_test).mean()

hybrid = rule_preds.where(rule_preds != "Other", predictions)
hybrid_accuracy = (hybrid == y_test).mean()

print(f"Rules: {rule_accuracy:.0%}")
print(f"Model: {accuracy:.0%}")
print(f"Hybrid: {hybrid_accuracy:.0%}")

In [ ]:
mistakes = test[hybrid != y_test]
mistakes[["category"]].assign(predicted=hybrid[hybrid != y_test])

In [ ]:
review = test[["id", "description", "category"]].copy()
review["rules"] = rule_preds
review["model"] = predictions
review["hybrid"] = hybrid

review[review["hybrid"] != review["category"]]

In [ ]:
most_common = y_train.value_counts().idxmax()
dumb_accuracy = (y_test == most_common).mean()
print(f"Always guessing {most_common}: {dumb_accuracy:.0%}")

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, predictions, zero_division=0))

In [ ]:
fixed = df[(df["category_source"] == "manual") & (df["category"].isin(["Subscriptions", "Utilities"]))]
fixed[["description", "category"]]

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

pipeline = make_pipeline(CountVectorizer(), MultinomialNB())

X = labelled["description"]
y = labelled["category"]
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model_scores = cross_val_score(pipeline, X, y, cv=folds)

dumb = make_pipeline(CountVectorizer(), DummyClassifier(strategy="most_frequent"))
dumb_scores = cross_val_score(dumb, X, y, cv=folds)

print("Model scores:", model_scores.round(2))
print(f"Model: {model_scores.mean():.0%} +- {model_scores.std():.0%}")
print(f"Dumb baseline: {dumb_scores.mean():.0%}")

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

groups = labelled["description"].str.split().str[0]

group_folds = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
group_scores = cross_val_score(pipeline, X, y, cv=group_folds, groups=groups)
dumb_group_scores = cross_val_score(dumb, X, y, cv=group_folds, groups=groups)

print("Unseed-merchant scores:", group_scores.round(2))
print(f"Model: {group_scores.mean():.0%} +- {group_scores.std():.0%}")
print(f"Dumb baseline: {dumb_group_scores.mean():.0%}")

In [ ]:
char_pipeline = make_pipeline(
    CountVectorizer(analyzer="char_wb", ngram_range=(3, 5)),
    MultinomialNB(),
)

char_scores = cross_val_score(char_pipeline, X, y, cv=group_folds, groups=groups)

print("Word scores:", group_scores.round(2))
print("Char scores:", char_scores.round(2))
print(f"Word model: {group_scores.mean():.0%} +- {group_scores.std():.0%}")
print(f"Char model: {char_scores.mean():.0%} +- {char_scores.std():.0%}")

In [ ]:
char_pipeline.fit(X, y)
len(char_pipeline.named_steps["countvectorizer"].vocabulary_)